In [37]:
# Setup and Dependencies
import os
import json
import torch

In [ ]:
# Core Data Transformation Functions
def load_json_to_tensor(filepath, expected_landmarks=33):
    """
    Reads a JSON file containing MediaPipe landmarks, sanitizes missing data,
    and converts it into a uniform 3D PyTorch tensor.
    """
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Handle both List and Dictionary JSON structures
    if isinstance(data, list):
        frames_data = data
    elif isinstance(data, dict):
        frames_data = data.get('frames', [])
    else:
        frames_data = []

    if not frames_data:
        print(f"No frames found in {filepath}")
        return None
        
    # Ensure frames are strictly in chronological order
    frames_data = sorted(frames_data, key=lambda x: x.get('frame_index', 0))
    
    all_frames_landmarks = []
    for frame in frames_data:
        landmarks = frame.get('landmarks', [])
        
        # --- THE FIX: Handle explicitly null landmarks array ---
        if landmarks is None:
            landmarks = []
            
        clean_landmarks = []
        
        # Enforce exactly 33 landmarks per frame
        for i in range(expected_landmarks):
            lm = landmarks[i] if i < len(landmarks) else None
            
            if lm is None:
                # Missing landmark -> fill with 4 zeros
                clean_landmarks.append([0.0, 0.0, 0.0, 0.0])
            else:
                # Sanitize coordinates, enforce exactly 4 features (x,y,z,v)
                clean_lm = []
                for j in range(4):
                    val = lm[j] if j < len(lm) else 0.0
                    clean_lm.append(0.0 if val is None else val)
                clean_landmarks.append(clean_lm)
                
        all_frames_landmarks.append(clean_landmarks)
        
    try:
        # Expected shape: [Frames, 33, 4]
        return torch.tensor(all_frames_landmarks, dtype=torch.float32)
    except Exception as e:
        print(f"Skipping {filepath} due to unrecoverable data corruption: {e}")
        return None


def flatten_spatial_tensor(tensor):
    """
    Flattens spatial topology [Frames, Landmarks, Features] 
    into a 1D sequence per frame [Frames, Features*Landmarks].
    """
    frames, landmarks, features = tensor.shape
    flat_size = landmarks * features  # 33 * 4 = 132
    flattened_tensor = tensor.view(frames, flat_size)
    return flattened_tensor, flat_size

In [39]:
#  Batch Processing Logic
def process_dataset(input_base_dir, output_base_dir, exercise_classes):
    """
    Iterates through all exercise folders, processes every JSON file, 
    and saves the flattened .pt files to the output directory.
    """
    total_processed = 0
    
    for exercise in exercise_classes:
        input_dir = os.path.join(input_base_dir, exercise)
        output_dir = os.path.join(output_base_dir, exercise)
        
        # Create output directory 
        os.makedirs(output_dir, exist_ok=True)
        
        # Skip if the input folder doesn't exist yet
        if not os.path.exists(input_dir):
            print(f"Warning: Input folder not found at {input_dir}. Skipping...")
            continue
            
        print(f"Processing '{exercise}' files...")
        file_count = 0
        
        # Iterate through every JSON file in the specific exercise folder
        for filename in os.listdir(input_dir):
            if filename.endswith('.json'):
                input_filepath = os.path.join(input_dir, filename)
                
                # Load JSON into a 3D Tensor (Now safe from jagging/None)
                spatial_tensor = load_json_to_tensor(input_filepath)
                if spatial_tensor is None:
                    continue
                    
                # Flatten the Tensor
                flattened_tensor, flat_size = flatten_spatial_tensor(spatial_tensor)
                
                # Save as a PyTorch .pt file
                output_filename = filename.replace('.json', '.pt')
                output_filepath = os.path.join(output_dir, output_filename)
                
                torch.save(flattened_tensor, output_filepath)
                file_count += 1
                total_processed += 1
                
        print(f"  -> Successfully flattened and saved {file_count} files for '{exercise}'.")
        if file_count > 0:
            print(f"  -> Final Tensor Shape for this class: {flattened_tensor.shape}\n")
            
    print(f"=== Pipeline Complete! Processed {total_processed} total files. ===")

In [40]:
# Execution Block

# Define project paths relative to the notebook location
INPUT_BASE = "../data/interpolated_json/"
OUTPUT_BASE = "../data/flattened_tensors/"

# The 4 exercise classes 
EXERCISES = ["squat", "push_up", "bench_press", "pull_up"]

# Run the pipeline
print("Starting Data Flattening Pipeline...")
process_dataset(INPUT_BASE, OUTPUT_BASE, EXERCISES)

Starting Data Flattening Pipeline...
Processing 'squat' files...
  -> Successfully flattened and saved 250 files for 'squat'.
  -> Final Tensor Shape for this class: torch.Size([138, 132])

Processing 'push_up' files...


TypeError: object of type 'NoneType' has no len()